In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/House_Prices.csv")
df = df.iloc[:1460].copy()

In [ ]:
# remove outliers that has higher GrLivArea with less saleprice
df = df.drop(index=[1298, 523])

df = df.drop(columns="Id")
df["MSSubClass"] = df["MSSubClass"].astype("str")

# fill missing values in categorical columns with none
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageQual", "GarageCond", "GarageFinish", "GarageType",
    "BsmtCond", "BsmtQual", "BsmtFinType1"
]
df[none_cols] = df[none_cols].fillna("None")

# fill missing values in num columns with 0
zero_cols = [
    "BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1",
    "GarageArea", "GarageCars"
]
df[zero_cols] = df[zero_cols].fillna(0)

# if the garage built year greater than the sold year mark the values as missing
df.loc[df["GarageYrBlt"] > df["YrSold"], "GarageYrBlt"] = np.nan

In [2]:
df["TotalArea"] = df[
    ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
].sum(axis=1, skipna=True)

In [3]:
df["TotalBathrooms"] = (
    df["FullBath"].fillna(0)
    + 0.5 * df["HalfBath"].fillna(0)
    + df["BsmtFullBath"].fillna(0)
    + 0.5 * df["BsmtHalfBath"].fillna(0)
)

TotalBathrooms combines full and half bathrooms from both above-ground and basement levels into one feature. Half bathrooms are weighted as 0.5.

In [4]:
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

HouseAge represents how old the house was when it was sold. It is calculated as YrSold - YearBuilt.

In [5]:
df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

YearsSinceRemodel represents how many years passed between the last renovation and the sale of the house.

In [6]:
df.loc[df["HouseAge"] < 0, "HouseAge"] = np.nan
df.loc[df["YearsSinceRemodel"] < 0, "YearsSinceRemodel"] = np.nan

In [7]:
df["TotalPorchArea"] = (
    df["WoodDeckSF"]
    + df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

TotalPorchArea combines the main porch and deck surface areas into one feature representing the property's total outdoor living area.

checking new features

In [8]:
engineered_features = [
    "TotalArea",
    "TotalBathrooms",
    "HouseAge",
    "YearsSinceRemodel",
    "TotalPorchArea"
]

df[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
TotalArea,1460.0,2567.048630,821.714421,334.0,2009.5,2474.0,3004.0,11752.0
TotalBathrooms,1460.0,2.210616,0.785399,1.0,2.0,2.0,2.5,6.0
HouseAge,1460.0,36.547945,30.250152,0.0,8.0,35.0,54.0,136.0
YearsSinceRemodel,1459.0,22.966415,20.638195,0.0,4.0,14.0,41.0,60.0
TotalPorchArea,1460.0,181.329452,156.656097,0.0,45.0,164.0,266.0,1027.0


In [9]:
df[engineered_features].isna().sum()

TotalArea            0
TotalBathrooms       0
HouseAge             0
YearsSinceRemodel    1
TotalPorchArea       0
dtype: int64

In [10]:
for col in engineered_features:
    print(col, (df[col] < 0).sum())

TotalArea 0
TotalBathrooms 0
HouseAge 0
YearsSinceRemodel 0
TotalPorchArea 0


integrate the engineered features into the preprocessing flow

In [12]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [17]:
print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

Train shape: (1168, 304)
Test shape: (292, 304)


In [18]:
print("Engineered features:")
print(engineered_features)

Engineered features:
['TotalArea', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel', 'TotalPorchArea']
